# 🧠 Chapter 01: How LLMs Actually Work — Transformers Intuition

---

## The Most Important Concept in All of GenAI 🎯

In the last notebook, we learned WHAT LLMs do. Now we learn HOW they do it.

**This is the #1 most asked topic in GenAI interviews.** Everyone can use ChatGPT — but can you explain *how it thinks*?

Don't worry — we'll use **intuition, not math**. By the end, you'll understand:
- How words become numbers (Embeddings)
- How the model "reads" text (Self-Attention)
- How it learns (Training Pipeline)
- Why it sometimes gets things wrong (Hallucinations)

---

## 🎭 The Story: How Does an LLM "Read"?

Imagine you're reading this sentence:

> *"The bank near the river was flooded after the storm."*

When you read the word **"bank"**, how do you know it means the *side of a river* and not a *financial institution*?

**Because you looked at the word "river" nearby!**

```
┌─────────────────────────────────────────────────────────────────────┐
│  "The bank near the river was flooded after the storm"              │
│                                                                     │
│       bank ←──── looks at ────→ river                              │
│                                                                     │
│  "Ah! 'river' is nearby, so 'bank' means riverbank, not money!"   │
│                                                                     │
│  This is EXACTLY what Self-Attention does!                         │
│  Every word looks at every other word to understand context.       │
└─────────────────────────────────────────────────────────────────────┘
```

This ability to understand context is the **core innovation** of the Transformer architecture.

---

## 📦 Step 1: Words → Numbers (Tokenization + Embeddings)

Computers can't understand words. They need numbers. So we need to convert text to numbers in a *meaningful way*.

This happens in two stages:

```
╔═══════════════════════════════════════════════════════════════════════╗
║                STAGE 1: TOKENIZATION                                  ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  "I love Python" → ["I", " love", " Python"]  → [40, 3476, 14820]  ║
║                                                                       ║
║  Each chunk gets a unique ID number (from a vocabulary)              ║
║  These IDs are arbitrary — "love" could be 3476 or 9999             ║
║  At this stage, the model has NO idea what words MEAN               ║
║                                                                       ║
╠═══════════════════════════════════════════════════════════════════════╣
║                STAGE 2: EMBEDDINGS                                    ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  Token IDs → Dense vectors (lists of numbers) that capture MEANING  ║
║                                                                       ║
║  "king"  → [0.21, -0.08, 0.93, 0.44, ...]  (768 or 1536 numbers)  ║
║  "queen" → [0.19, -0.07, 0.91, 0.52, ...]  (very similar!)        ║
║  "table" → [0.73, 0.62, -0.21, 0.15, ...]  (very different!)      ║
║                                                                       ║
║  NOW the model can work with meaning!                                ║
║  Similar concepts have similar numbers (nearby in vector space)     ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

---

### 💡 The Magic of Embeddings: King - Man + Woman = Queen

This is the famous example that blew everyone's mind:

```
┌─────────────────────────────────────────────────────────────────────┐
│                                                                     │
│   Think of embeddings as coordinates in a "meaning space"          │
│                                                                     │
│            Queen 👑                                                 │
│              ●                                                      │
│             /|                                                      │
│            / |  ←── Same "royalty" direction                       │
│           /  |                                                      │
│    King  ●   |                                                      │
│    👑    |   |                                                      │
│         |   ● Woman                                                │
│         |  /                                                       │
│         | / ←── Same "gender" direction                            │
│         |/                                                          │
│         ● Man                                                      │
│                                                                     │
│   Vector(King) - Vector(Man) + Vector(Woman) ≈ Vector(Queen)       │
│                                                                     │
│   The model learned that the relationship between                  │
│   king→queen is the same as man→woman!                             │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

Let's see this in real code!

In [ ]:
# Let's SEE embeddings in action!
from openai import OpenAI
import numpy as np
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

def get_embedding(text):
    """Get the embedding vector for a piece of text."""
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return np.array(response.data[0].embedding)

def cosine_similarity(a, b):
    """How similar are two vectors? (1.0 = identical, 0.0 = unrelated)"""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Let's compare some words!
words = ["happy", "joyful", "sad", "banana", "Python", "JavaScript"]
embeddings = {word: get_embedding(word) for word in words}

print("SEMANTIC SIMILARITY (1.0 = identical meaning)")
print("=" * 55)

pairs = [
    ("happy", "joyful"),     # Should be HIGH (similar meaning)
    ("happy", "sad"),        # Should be MEDIUM (same category: emotions)
    ("happy", "banana"),     # Should be LOW (unrelated)
    ("Python", "JavaScript"), # Should be HIGH (same category: programming)
    ("Python", "banana"),    # Should be LOW (different meanings)
]

for w1, w2 in pairs:
    sim = cosine_similarity(embeddings[w1], embeddings[w2])
    bar = "█" * int(sim * 40)
    print(f"  {w1:12s} ↔ {w2:12s} = {sim:.3f} {bar}")

**What did you notice?**

- "happy" and "joyful" are very similar (high cosine similarity)
- "happy" and "banana" are unrelated (low similarity)
- The model understands **meaning**, not just spelling!

---

In [ ]:
# Let's build a simple SEMANTIC SEARCH!
# This is the foundation of RAG (Chapter 5)

# Our "knowledge base" — imagine these are documents
documents = [
    "Python is a programming language used for AI and web development",
    "Machine learning allows computers to learn from data",
    "The Eiffel Tower is located in Paris, France",
    "Neural networks are inspired by the human brain",
    "Photosynthesis converts sunlight into energy for plants",
    "Deep learning uses multiple layers of neural networks",
    "The Great Wall of China is over 13,000 miles long",
    "Natural language processing helps computers understand text",
]

# Embed all documents
doc_embeddings = [get_embedding(doc) for doc in documents]

# Now search!
query = "How do AI systems learn?"
query_embedding = get_embedding(query)

# Calculate similarity with each document
similarities = [cosine_similarity(query_embedding, doc_emb) for doc_emb in doc_embeddings]

# Sort by similarity
results = sorted(zip(similarities, documents), reverse=True)

print(f"🔍 Query: \"{query}\"")
print("=" * 65)
print("\nTop results (by semantic similarity):")
for i, (sim, doc) in enumerate(results[:3]):
    print(f"  {i+1}. [{sim:.3f}] {doc}")

print("\n🎉 It found AI-related documents even though the exact words differ!")
print("   This is the MAGIC of embeddings — semantic search, not keyword search!")

**💥 That's a mini-RAG system!** We'll build a full one in Chapter 5.

---

## 🔍 Step 2: Self-Attention — How LLMs "Read"

Now the critical part. How does the model understand the *relationships* between words?

```
╔═══════════════════════════════════════════════════════════════════════╗
║                    SELF-ATTENTION (Intuition)                         ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  Analogy: Speed Reading with Highlighters 📚🖍️                       ║
║                                                                       ║
║  Imagine you're reading a book, and for EACH word, you have         ║
║  unlimited highlighters. You highlight which OTHER words             ║
║  are most important for understanding THIS word.                     ║
║                                                                       ║
║  ─────────────────────────────────────────────────────────────────   ║
║                                                                       ║
║  Sentence: "The cat sat on the mat because it was tired"             ║
║                                                                       ║
║  When processing "it":                                                ║
║  ┌────────────────────────────────────────────────────────────────┐ ║
║  │  The   cat   sat   on   the   mat   because   it   was  tired │ ║
║  │  ░░░   ████  ░░░  ░░░  ░░░   ░░░   ░░░      ████  ░░░  ░░░  │ ║
║  │        ↑↑↑↑                                  ↑↑↑↑            │ ║
║  │    HIGH attention                        (itself)             │ ║
║  │                                                               │ ║
║  │  "it" pays most attention to "cat" → so "it" = the cat!      │ ║
║  └────────────────────────────────────────────────────────────────┘ ║
║                                                                       ║
║  Every word does this simultaneously (that's the "parallel" power)  ║
║  This is why Transformers are so fast compared to older models!     ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

### Why is this revolutionary?

**Before Transformers (RNNs, LSTMs):**
- Processed words ONE AT A TIME, left to right
- By the time you reach word 100, you've "forgotten" word 1
- Like reading through a straw 🥤

**With Transformers (Self-Attention):**
- EVERY word sees EVERY other word, all at once
- No forgetting! Word 1 and word 100 are equally accessible
- Like reading with panoramic vision 👓

---

## 🤔 Quick Check: Are You Following?

**Sentence:** "The doctor told the nurse that she should take a break."

**Question:** Who should take a break — the doctor or the nurse?

<details>
<summary>👆 Click to see answer</summary>

Most humans say **the nurse** (because of context: the doctor *told* the nurse).

An LLM figures this out using self-attention! The word **"she"** attends heavily to **"nurse"** because:
1. "nurse" is a common referent in this structure
2. The verb "told ... that she should" implies the instruction is FOR the nurse

This is called **coreference resolution** — and Transformers do it naturally through attention!

</details>

---

## 🏭 Step 3: The Transformer Architecture (Big Picture)

Let's zoom out and see the full pipeline:

```
╔═══════════════════════════════════════════════════════════════════════╗
║               THE TRANSFORMER (10,000-foot view)                     ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  INPUT: "The capital of France is"                                    ║
║                │                                                      ║
║                ▼                                                      ║
║  ┌───────────────────────────────┐                                   ║
║  │      1. TOKENIZE              │  Split text into tokens           ║
║  │  ["The", "capital", "of",      │  Each gets a unique ID           ║
║  │   "France", "is"]             │                                   ║
║  └──────────────┬────────────────┘                                   ║
║                 ▼                                                     ║
║  ┌───────────────────────────────┐                                   ║
║  │      2. EMBED                 │  Convert IDs to meaning          ║
║  │  Each token → vector of       │  vectors (1536 numbers)          ║
║  │  numbers (+ position info)    │                                   ║
║  └──────────────┬────────────────┘                                   ║
║                 ▼                                                     ║
║  ┌───────────────────────────────┐                                   ║
║  │   3. SELF-ATTENTION × N       │  Each word looks at every        ║
║  │  (Repeated 32-96 times!)      │  other word. "France" and        ║
║  │                               │  "capital" attend to each        ║
║  │  This is where the model      │  other strongly!                 ║
║  │  THINKS and UNDERSTANDS       │                                   ║
║  └──────────────┬────────────────┘                                   ║
║                 ▼                                                     ║
║  ┌───────────────────────────────┐                                   ║
║  │   4. PREDICT NEXT TOKEN       │  Output a probability over       ║
║  │                               │  ALL possible tokens:             ║
║  │  "Paris"  → 94.2%             │                                   ║
║  │  "Lyon"   → 2.1%             │  Pick the most likely one!       ║
║  │  "Berlin" → 0.3%             │                                   ║
║  └───────────────────────────────┘                                   ║
║                                                                       ║
║  OUTPUT: "Paris" ✅                                                   ║
║                                                                       ║
║  Then append "Paris" to input and predict NEXT token...              ║
║  "The capital of France is Paris _" → "." → STOP                    ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

---

## 🎓 Step 4: How LLMs Learn (Training Pipeline)

An LLM goes through **three stages** of training:

```
╔═══════════════════════════════════════════════════════════════════════╗
║             THE THREE STAGES OF LLM TRAINING                         ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  STAGE 1: PRE-TRAINING 📚                                            ║
║  ┌──────────────────────────────────────────────────────────────┐   ║
║  │  • Read the ENTIRE internet (books, Wikipedia, code, etc.)   │   ║
║  │  • Task: Predict the next word, billions of times            │   ║
║  │  • Learns: Language patterns, facts, reasoning               │   ║
║  │  • Cost: $1M – $100M+ (thousands of GPUs for months)         │   ║
║  │  • Result: "Base model" — smart but unhelpful                │   ║
║  │    (It just completes text, doesn't follow instructions)     │   ║
║  └──────────────────────────────────────────────────────────────┘   ║
║                          ↓                                           ║
║  STAGE 2: INSTRUCTION TUNING (SFT) 🎯                                ║
║  ┌──────────────────────────────────────────────────────────────┐   ║
║  │  • Show the model examples of instructions + good answers    │   ║
║  │  • "Q: Summarize this article" → Model learns to summarize   │   ║
║  │  • Human annotators create high-quality examples              │   ║
║  │  • Cost: $10K – $1M                                          │   ║
║  │  • Result: Helpful assistant (but may still be unsafe)       │   ║
║  └──────────────────────────────────────────────────────────────┘   ║
║                          ↓                                           ║
║  STAGE 3: RLHF / RLAIF (Alignment) 🛡️                                ║
║  ┌──────────────────────────────────────────────────────────────┐   ║
║  │  • Humans (or AI) rate which responses are BETTER            │   ║
║  │  • Model learns: "Users prefer response A over B"            │   ║
║  │  • This makes it helpful, harmless, and honest               │   ║
║  │  • RLHF = Reinforcement Learning from Human Feedback         │   ║
║  │  • RLAIF = RL from AI Feedback (cheaper, used in 2026)       │   ║
║  │  • Result: ChatGPT, Claude — safe, aligned, useful           │   ║
║  └──────────────────────────────────────────────────────────────┘   ║
║                                                                       ║
║  💡 As engineers, WE work at the TOP of this stack:                   ║
║     We use pre-trained models and customize them with                ║
║     prompts, RAG, fine-tuning, and orchestration frameworks.        ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

---

## 🏥 Mixture of Experts (MoE) — The Modern Architecture

Most modern LLMs (GPT-4, Mixtral) use a technique called **Mixture of Experts**:

```
╔═══════════════════════════════════════════════════════════════════════╗
║                MIXTURE OF EXPERTS (MoE)                               ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  Analogy: A Hospital 🏥                                               ║
║                                                                       ║
║  Instead of one doctor who knows everything (impossible!),           ║
║  you have a hospital with SPECIALISTS:                               ║
║                                                                       ║
║  ┌────────────────────────────────────────────────────────────────┐ ║
║  │                        🧭 ROUTER                              │ ║
║  │              (Decides which experts to use)                    │ ║
║  │                    ┌────┼────┐                                 │ ║
║  │                    ▼    ▼    ▼                                 │ ║
║  │  ┌─────────┐ ┌─────────┐ ┌─────────┐ ┌─────────┐            │ ║
║  │  │Expert 1 │ │Expert 2 │ │Expert 3 │ │Expert 8 │            │ ║
║  │  │ Code 💻 │ │ Math 🔢│ │Poetry 📝│ │Science🔬│  ...       │ ║
║  │  └─────────┘ └─────────┘ └─────────┘ └─────────┘            │ ║
║  │       ✅          ✅           ❌           ❌                │ ║
║  │  (activated)  (activated) (not needed) (not needed)           │ ║
║  └────────────────────────────────────────────────────────────────┘ ║
║                                                                       ║
║  For a coding question: Only the "code" and "math" experts fire!   ║
║  For a poem: Only the "poetry" and "language" experts fire!         ║
║                                                                       ║
║  Result: Model has 1 TRILLION parameters total, but only uses      ║
║  ~200 BILLION at any time → MUCH faster and cheaper to run!        ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

---

## 😵 Why LLMs Hallucinate (And What We Can Do About It)

This is a **critical interview topic:**

```
╔═══════════════════════════════════════════════════════════════════════╗
║                    WHY LLMs HALLUCINATE                               ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  ROOT CAUSE:                                                          ║
║  LLMs generate STATISTICALLY PROBABLE text, not VERIFIED facts.     ║
║                                                                       ║
║  It's like asking someone who read 1000 history books 5 years ago   ║
║  to recall exact dates — they'll sometimes "fill in" with           ║
║  plausible-sounding but wrong information.                           ║
║                                                                       ║
║  TYPES OF HALLUCINATION:                                              ║
║  ┌──────────────────────────────────────────────────────────────┐   ║
║  │  1. Factual errors  │  "Gandhi was born in 1875"            │   ║
║  │                     │  (Actually: 1869)                      │   ║
║  │                     │                                        │   ║
║  │  2. Fabrication     │  "According to the 2024 study by      │   ║
║  │                     │   Dr. Smith published in Nature..."    │   ║
║  │                     │  (This study doesn't exist!)           │   ║
║  │                     │                                        │   ║
║  │  3. Inconsistency   │  Says X in paragraph 1, then          │   ║
║  │                     │  contradicts X in paragraph 3          │   ║
║  └──────────────────────────────────────────────────────────────┘   ║
║                                                                       ║
║  MITIGATIONS (We'll learn ALL of these in this course!):             ║
║  ┌──────────────────────────────────────────────────────────────┐   ║
║  │  1. RAG (Ch 5)         │  Give the model REAL documents     │   ║
║  │  2. Tool Use (Ch 6)    │  Let it search/calculate           │   ║
║  │  3. Guardrails (Ch 10) │  Check output for correctness     │   ║
║  │  4. Temperature=0      │  Reduce randomness                 │   ║
║  │  5. System prompts     │  "Only answer based on context"    │   ║
║  └──────────────────────────────────────────────────────────────┘   ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

---

## 🧪 Hands-On: Visualize Embeddings in 2D

In [ ]:
# Let's visualize how embeddings cluster similar concepts!
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Words from different categories
categories = {
    "Animals 🐾": ["cat", "dog", "elephant", "fish", "bird"],
    "Programming 💻": ["Python", "JavaScript", "algorithm", "database", "API"],
    "Food 🍕": ["pizza", "sushi", "pasta", "burger", "salad"],
}

all_words = []
all_embeddings = []
all_labels = []

for category, words in categories.items():
    for word in words:
        all_words.append(word)
        all_embeddings.append(get_embedding(word))
        all_labels.append(category)

# Reduce from 1536 dimensions to 2 for visualization
pca = PCA(n_components=2)
coords = pca.fit_transform(all_embeddings)

# Plot!
colors = {"Animals 🐾": "#FF6B6B", "Programming 💻": "#4ECDC4", "Food 🍕": "#FFE66D"}

plt.figure(figsize=(12, 8))
plt.style.use('dark_background')

for i, (word, coord) in enumerate(zip(all_words, coords)):
    color = colors[all_labels[i]]
    plt.scatter(coord[0], coord[1], c=color, s=100, zorder=5)
    plt.annotate(word, (coord[0]+0.01, coord[1]+0.01), fontsize=11, color=color)

# Add legend
for category, color in colors.items():
    plt.scatter([], [], c=color, s=100, label=category)
plt.legend(fontsize=12, loc='upper left')

plt.title("Embeddings in 2D — Similar concepts cluster together!", fontsize=14)
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.tight_layout()
plt.show()

print("\n🎉 Notice how words from the SAME category cluster together!")
print("   Even though no one told the model about categories!")
print("   The model LEARNED this structure from reading text.")

---

## 📝 Quick Quiz: Check Your Understanding

---

### Question 1:
What is the purpose of self-attention in Transformers?

- A) To make the model run faster
- B) To let each word understand its context by looking at all other words
- C) To compress the text into fewer tokens
- D) To translate between languages

<details><summary>Answer</summary>

**B) To let each word understand its context by looking at all other words**

Self-attention computes a relevance score between every pair of tokens. For 
"The bank near the river", the token "bank" attends strongly to "river" to understand it means "riverbank" not "financial institution."

</details>

---

### Question 2:
What are embeddings?

- A) Compressed versions of images
- B) Dense numerical vectors that capture the meaning of text
- C) Encrypted versions of passwords
- D) HTML tags embedded in web pages

<details><summary>Answer</summary>

**B) Dense numerical vectors that capture the meaning of text**

Each word/token is represented as a vector (list of numbers, typically 768-3072 dimensions). Similar meanings produce similar vectors: cos_sim("happy", "joyful") > cos_sim("happy", "table").

</details>

---

### Question 3:
What does RLHF stand for, and why is it important?

- A) Random Learning with High Frequency — speeds up training
- B) Reinforcement Learning from Human Feedback — makes models helpful and safe
- C) Real-time Language Hosting Framework — deploys models
- D) Recursive Layer for Hidden Features — a neural network technique

<details><summary>Answer</summary>

**B) Reinforcement Learning from Human Feedback — makes models helpful and safe**

After pre-training and instruction tuning, RLHF is the final step that aligns the model with human preferences. Humans rate which responses are better, and the model learns to produce preferred responses. This is what turns a raw text predictor into a helpful, harmless assistant (like ChatGPT).

</details>

---

### Question 4:
What is Mixture of Experts (MoE)?

- A) Using multiple different LLMs to answer the same question
- B) An architecture where only a subset of parameters are active per token
- C) Hiring multiple human experts to check AI outputs
- D) A way to merge different datasets for training

<details><summary>Answer</summary>

**B) An architecture where only a subset of parameters are active per token**

MoE allows models to have many parameters (e.g., 1 trillion) but only activate a fraction (e.g., 200 billion) for each input. A router decides which "expert" sub-networks to use. This makes the model powerful yet efficient — the hospital with specialists analogy!

</details>

---

## 💡 Interview Prep: Common Questions

```
╔═══════════════════════════════════════════════════════════════════════╗
║               INTERVIEW QUESTIONS FOR THIS MODULE                    ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  Q: Explain the Transformer architecture in simple terms.            ║
║  A: Input text is tokenized, converted to embedding vectors,        ║
║     then passed through multiple self-attention layers where         ║
║     every token attends to every other token to build contextual    ║
║     understanding. Finally, the model predicts the next token       ║
║     as a probability distribution over the vocabulary.               ║
║                                                                       ║
║  Q: What is self-attention? Why is it important?                      ║
║  A: Self-attention lets each token compute a weighted sum of all    ║
║     other tokens, where the weights represent relevance. This       ║
║     captures long-range dependencies ("it" refers to "cat"          ║
║     even if they're 20 words apart). It replaced sequential         ║
║     processing (RNNs) with parallel processing — huge speedup.     ║
║                                                                       ║
║  Q: What are embeddings? How do they capture meaning?                ║
║  A: Embeddings are learned numerical representations where close    ║
║     vectors = close meanings. They're learned during training:      ║
║     words that appear in similar contexts get similar vectors.       ║
║     This enables semantic search, analogies, and understanding.     ║
║                                                                       ║
║  Q: Explain the training pipeline of a modern LLM.                   ║
║  A: 1) Pre-training: Next-token prediction on internet-scale data  ║
║     2) SFT: Supervised fine-tuning on instruction-response pairs    ║
║     3) RLHF/RLAIF: Alignment using human or AI preference data     ║
║                                                                       ║
║  Q: Why do LLMs hallucinate? How do you mitigate it?                ║
║  A: They're probabilistic generators, not knowledge databases.      ║
║     Mitigations: RAG (ground in real data), low temperature,        ║
║     tool use (search/calculate), output guardrails, and             ║
║     prompting strategies ("only answer based on provided context"). ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

---

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 Tokens → Embeddings → Self-Attention → Next Token Prediction  │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Embeddings turn words into meaningful number vectors           │
│     Similar meanings = nearby in vector space                      │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Self-Attention lets every word see every other word            │
│     This is what makes Transformers so powerful!                   │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Training: Pre-train → Instruction Tune → RLHF                 │
├─────────────────────────────────────────────────────────────────────┤
│  📌 MoE = Multiple experts, only few active at a time             │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Hallucinations happen because LLMs are generators, not DBs    │
│     Mitigate with: RAG, tools, guardrails, temperature=0          │
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Prompt Engineering!

Now that you understand HOW LLMs work, let's learn the art and science of **talking to them effectively** — Chain-of-Thought, ReAct, Structured Outputs, and more!

---

### 🎉 Incredible Progress!
You now understand the Transformer architecture better than 90% of GenAI users. This knowledge will be your **superpower** in interviews! 🚀